In [1]:
# 测试雪花算法生成的ID的时间戳差值
config_file = "1.txt"
with open(config_file, "r", encoding="utf-8") as file:
    lines = file.readlines()
    num=0;
    prev_timestamp = 0
    for i, line in enumerate(lines):
        id = int(line.strip().split(" ")[1])
        # print(id)
        #TypeError: 'str' object is not callable
        binary = __builtins__.bin(id)[2:].zfill(64)
        # print(binary)
        timestamp = binary[1:43]
        timestamp = int(binary[1:43], 2)
        # print(timestamp)
        #打印每个timestamp的差值
        if i > 0:
            diff = timestamp - prev_timestamp
            print(diff)
        prev_timestamp = timestamp


172800524
172800502
172800526
172800494
172800514
172800488
172800510
172800478
172800480
172800518


In [4]:
import requests
import threading
import json
import time
from datetime import datetime, timedelta

import math




# 存储结果的字典，用于保存每个日期的预约信息
results = {}


# 预约接口的 URL
order_url = "http://www.ruanjiezh.cn:8081/api/mobile/order/create"
# 配置文件路径
config_file = "confignew.txt"
# 用户的 openId 列表
openIds = []
# 日期和时段的映射
datetimes = {}

# 用户名
name = ''
# 用户手机号
phone = ''
# 星期二的标识
Tuesday = ''
# 刷新间隔时间
interval = 0.2

# 读取配置文件，解析内容并赋值到对应变量 
# 读取配置到 datetimes 字典 {'2025-05-31': ['17', '18'], '2025-06-01': ['17']}
with open(config_file, "r", encoding="utf-8") as file:
    lines = file.readlines()
    for i, line in enumerate(lines):
        print(line)
        # 解析用户名
        if line.startswith("name"):
            name = line.strip().split("=")[1]
        
        # 解析手机号
        if line.startswith("phone"):
            phone = line.strip().split("=")[1]
        # 解析 openId 列表
        if line.startswith("openIds"):
            openIds = line.strip().split("=")[1].split(",")
        # 解析日期和时段映射
        if line.startswith("datetimes"):
            tdatetimes = line.strip().split("=")[1].split(",")
            for t in tdatetimes:
                print(t)
                day_t = t.strip().split("#")
                datetimes[day_t[0]] = day_t[1].split('@')


# 使用线程池来并发请求每个日期的预约信息
import requests
from concurrent.futures import ThreadPoolExecutor
# 定义线程池大小
THREAD_POOL_SIZE = 100
# 创建线程池
executor = ThreadPoolExecutor(max_workers=THREAD_POOL_SIZE)
# 定义请求函数
def fetch_reservation(d,t):
    """获取指定日期的预约信息"""
    global results
    firstId=0
    
    while True:
        # 构造请求 URL
        # print(f"正在获取 {d} 的预约信息...")
        url = f'http://www.ruanjiezh.cn:8081/api/mobile/reservation/tag/{d}'
        try:
            response = requests.get(url)
            if response.status_code == 200:
                # 将结果存入字典
                rjson = json.loads(response.text)
                status=rjson.get('status')
                if status:
                    res = rjson.get('data').get("items")
                    firstId=res[0].get("reservationId")
        except Exception as e:
            print(f"请求 {d} 时发生错误: {e}")
        if firstId != 0:
            break
    print(f"获取 {d} 的预约信息成功，firstId: {firstId}")
    ids =getIds(t,firstId)   
    print(f"日期 {d} 的预约信息: {ids}")     

     # 创建抢票实例
    order_client = MultiThreadOrder(order_url, openIds[0], name, phone)
    
    # 要预约的ID列表
    
    
    # 执行批量预约
    results = order_client.batch_order(ids, d)
    
    # 打印结果
    for i, result in enumerate(results):
        print(f"ID {ids[i]} 预约结果: {result}")  

    
# 提交任务到线程池
# 遍历 datetimes 字典中，每个日期一个线程请求当日场地信息
for d,t in datetimes.items():
    print(f"提交任务获取 {d} 的预约信息...")
    executor.submit(fetch_reservation, d,t)
# 等待所有线程完成
executor.shutdown(wait=True)





openIds=oIVg65rRnO0DVI-DHjR9ggZ29f5Y

datetimes=2025-06-02#17@18,2025-06-03#17

2025-06-02#17@18
2025-06-03#17
name=李

phone=13811872704

interval=1
提交任务获取 2025-06-02 的预约信息...
提交任务获取 2025-06-03 的预约信息...
获取 2025-06-02 的预约信息成功，firstId: 1373702962580299776
日期 2025-06-02 的预约信息: [1373702962580299816, 1373702962580299817, 1373702962580299818, 1373702962580299819, 1373702962580299820, 1373702962580299821, 1373702962580299822, 1373702962580299823]
预约请求发送: {'openId': 'oIVg65rRnO0DVI-DHjR9ggZ29f5Y', 'reservationId': '1373702962580299818', 'userName': '李', 'userPhone': '13811872704', 'date': '2025-06-02'}, 响应: {'status': False, 'msg': '该场地所选时间段已预订'}
预约请求发送: {'openId': 'oIVg65rRnO0DVI-DHjR9ggZ29f5Y', 'reservationId': '1373702962580299816', 'userName': '李', 'userPhone': '13811872704', 'date': '2025-06-02'}, 响应: {'status': False, 'msg': '该场地所选时间段已预订'}
预约请求发送: {'openId': 'oIVg65rRnO0DVI-DHjR9ggZ29f5Y', 'reservationId': '1373702962580299823', 'userName': '李', 'userPhone': '13811872704', 'date': '2025-

KeyboardInterrupt: 

预约请求发送: {'openId': 'oIVg65rRnO0DVI-DHjR9ggZ29f5Y', 'reservationId': '1373702962580299821', 'userName': '李', 'userPhone': '13811872704', 'date': '2025-06-02'}, 响应: {'status': False, 'msg': '该场地所选时间段已预订'}预约请求发送: {'openId': 'oIVg65rRnO0DVI-DHjR9ggZ29f5Y', 'reservationId': '1373702962580299823', 'userName': '李', 'userPhone': '13811872704', 'date': '2025-06-02'}, 响应: {'status': False, 'msg': '该场地所选时间段已预订'}

预约请求发送: {'openId': 'oIVg65rRnO0DVI-DHjR9ggZ29f5Y', 'reservationId': '1373702962580299816', 'userName': '李', 'userPhone': '13811872704', 'date': '2025-06-02'}, 响应: {'status': False, 'msg': '该场地所选时间段已预订'}
预约请求发送: {'openId': 'oIVg65rRnO0DVI-DHjR9ggZ29f5Y', 'reservationId': '1373702962580299820', 'userName': '李', 'userPhone': '13811872704', 'date': '2025-06-02'}, 响应: {'status': False, 'msg': '该场地所选时间段已预订'}
预约请求发送: {'openId': 'oIVg65rRnO0DVI-DHjR9ggZ29f5Y', 'reservationId': '1373702962580299817', 'userName': '李', 'userPhone': '13811872704', 'date': '2025-06-02'}, 响应: {'status': False, 'msg': 

In [5]:
from datetime import datetime
import time

start='00:00:00'
while True:
    now = datetime.now()
    current_time = now.strftime("%H:%M:%S")
    
    if current_time >= start:
        break
    # 每秒检查一次
    time.sleep(0.1)
print("开始预约...")

开始预约...


In [5]:
datetimes
# 遍历 datetimes 字典，打印每个日期和对应的时段
for date, times in datetimes.items():
    print(f"日期: {date}, 时段: {', '.join(times)}")


日期: 2025-05-31, 时段: 17, 18
日期: 2025-06-01, 时段: 17


In [1]:
#抢票日期
date1='2025-05-31'
#根据第一个日期获取需要抢票的id数组
def getIds(timeinday,first_playground):
    ids = []
    for t in timeinday:
        res=(int(t)-7)*4+int(first_playground)
        id=int(first_playground)+(int(t)-7)*4
        ids.append(id)
        ids.append(id+1)
        ids.append(id+2)
        ids.append(id+3)
    return ids
ids=getIds(t, '1371528630122848256');

NameError: name 't' is not defined

In [3]:
#抢票日期
date1='2025-05-31'
#根据第一个日期获取需要抢票的id数组
def getIds(timeinday,first_playground):
    ids = []
    for t in timeinday:
        res=(int(t)-7)*4+int(first_playground)
        id=int(first_playground)+(int(t)-7)*4
        ids.append(id)
        ids.append(id+1)
        ids.append(id+2)
        ids.append(id+3)
    return ids
ids=getIds(t, '1371528630122848256');

NameError: name 't' is not defined

In [ ]:
import threading
#传入ids列表，开启多个线程并发送请求
def order(ids):    
    #开启ids长度个线程
    
    for id in ids:
        thread = threading.Thread(target=lambda x: requests.post(order_url, json={
            "openId": openIds[0],
            "name": name,
            "phone": phone,
            "reservationId": x,
            "date": date1
        }), args=(id,))
        threads.append(thread)
        thread.start()
    # 等待所有线程完成
    for thread in threads:
        thread.join()
    print(f"抢票请求已发送，ID列表: {ids}")


In [2]:
import threading
import requests
import time
from concurrent.futures import ThreadPoolExecutor
from typing import List

class MultiThreadOrder:
    def __init__(self, order_url: str, openid: str, name: str, phone: str):
        self.order_url = order_url
        self.openid = openid
        self.name = name
        self.phone = phone
        
    def send_order(self, reservation_id: str, date: str) -> dict:
        """发送单个预约请求"""
        param = {
            "openId": self.openid,
            "reservationId": reservation_id,
            "userName": self.name,
            "userPhone": self.phone,
            "date": date
        }
        try:
            success = False
            while not success:
                response = requests.post(self.order_url, json=param)
                time.sleep(interval)
                res=response.json();
                success=res.get("status")
                print(f"预约请求发送: {param}, 响应: {res}")

            return response.json()
        except Exception as e:
            print(f"预约请求失败: {e}")
            return {"success": False, "error": str(e)}

    def batch_order(self, ids: List[str], date: str, max_workers: int = 50):
        """批量发送预约请求"""
        with ThreadPoolExecutor(max_workers=max_workers) as executor:
            futures = [
                executor.submit(self.send_order, str(id), date)
                for id in ids
            ]
            results = []
            for future in futures:
                try:
                    result = future.result()
                    results.append(result)
                except Exception as e:
                    print(f"线程执行失败: {e}")
                    results.append({"success": False, "error": str(e)})
        
        return results

# 使用示例
if __name__ == "__main__":
    # 配置信息
    ORDER_URL = "http://www.ruanjiezh.cn:8081/api/mobile/order/create"
    openid = "your_openid"
    name = "your_name"
    phone = "your_phone"
    
    # 创建抢票实例
    order_client = MultiThreadOrder(ORDER_URL, openid, name, phone)
    
    # 要预约的ID列表
    ids_to_order = ["1", "2", "3", "4"]  # 替换为实际的预约ID
    date = "2025-05-31"
    
    # 执行批量预约
    # results = order_client.batch_order(ids_to_order, date)
    
    # # 打印结果
    # for i, result in enumerate(results):
    #     print(f"ID {ids_to_order[i]} 预约结果: {result}")


In [ ]:
import os

def simple_write_appointment_info(date_str, appointment_ids):
    """
    简单版本：直接追加到文件末尾
    """
    # 构建输出内容
    ids_str = ','.join(map(str, appointment_ids))
    output_content = f"{date_str}={ids_str}\n"
    
    # 获取上一级目录路径
    parent_dir = os.path.dirname(os.getcwd())
    file_path = os.path.join(parent_dir, 'info.txt')
    
    try:
        # 以追加模式写入文件
        with open(file_path, 'a', encoding='utf-8') as f:
            f.write(output_content)
        
        print(f" 成功追加到文件: {file_path}")
        print(f" 写入内容: {output_content.strip()}")
        
    except Exception as e:
        print(f" 写入文件时出错: {e}")

In [10]:
import os
from pathlib import Path

def read_info_file_simple() -> dict:
    """
    简洁版本：读取info.txt文件并返回字典
    """
    file_path = Path(__file__).parent.parent / 'info.txt'
    
    if not file_path.exists():
        return {}
    
    result = {}
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line and '=' in line:
                date, ids_str = line.split('=', 1)
                result[date.strip()] = [id_str.strip() for id_str in ids_str.split(',')]
    
    return result

In [7]:
simple_write_appointment_info('2021-01-02', [1, 2, 3, 4])

✅ 成功追加到文件: /Users/donghongbin/consultant/info.txt
📝 写入内容: 2021-01-02=1,2,3,4


In [13]:
# 使用示例
appointment_data = read_info_file_simple()
print("简洁版本结果:")
for date, ids in appointment_data.items():
    print(f"{date}: {ids}")

简洁版本结果:
2025-08-19: ['1401969284690747408', '1401969284690747409', '1401969284690747410', '1401969284690747411']
2025-08-20: ['1402331673378430992', '1402331673378430993', '1402331673378430994', '1402331673378430995']
2025-08-23: ['1403418839294681104', '1403418839294681105', '1403418839294681106', '1403418839294681107']
2025-08-21: ['1402694062028365840', '1402694062028365841', '1402694062028365842', '1402694062028365843']
2025-08-24: ['1403781227961393168', '1403781227961393169', '1403781227961393170', '1403781227961393171']
2025-08-25: ['1404143616607133712', '1404143616607133713', '1404143616607133714', '1404143616607133715']
2025-08-22: ['1403056450648940560', '1403056450648940561', '1403056450648940562', '1403056450648940563']


In [ ]:
def read_info_file_simple() -> dict:
    """
    简洁版本：读取info.txt文件并返回字典
    """
    import os
    from pathlib import Path

    # 获取当前工作目录的上一级目录
    parent_dir = Path(os.getcwd()).parent
    file_path = parent_dir / 'info.txt'
    
    if not file_path.exists():
        return {}
    
    result = {}
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if line and '=' in line:
                date, ids_str = line.split('=', 1)
                result[date.strip()] = [id_str.strip() for id_str in ids_str.split(',')]
    
    return result

In [ ]:
import requests
from seatable_api import Base, context
import time
import logging

# 设置日志
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# 1. 认证和初始化 SeaTable 连接
SERVER_URL = context.server_url
API_TOKEN = context.api_token
base = Base(API_TOKEN, SERVER_URL)
base.auth()

def get_stock_price_tushare(stock_code):
    """
    使用 Tushare 获取股票价格
    需要先在 https://tushare.pro 注册获取 token
    """
    try:
        import tushare as ts
        # 设置你的 Tushare token（需要在脚本设置中配置环境变量）
        ts.set_token('你的TUSHARE_TOKEN')  
        
        pro = ts.pro_api()
        
        # 判断股票代码类型
        if stock_code.startswith(('0', '3')):
            ts_code = stock_code + '.SZ'
        elif stock_code.startswith('6'):
            ts_code = stock_code + '.SH'
        else:
            ts_code = stock_code
            
        # 获取实时行情
        df = pro.daily(ts_code=ts_code, trade_date=time.strftime('%Y%m%d'))
        
        if df.empty:
            # 如果当日无数据，获取最新数据
            df = pro.daily(ts_code=ts_code, limit=1)
            
        if not df.empty:
            return float(df['close'].iloc[0])
            
    except Exception as e:
        logger.warning(f"Tushare 接口失败: {e}")
    
    return None

def get_stock_price_akshare(stock_code):
    """
    使用 AkShare 获取股票价格
    """
    try:
        import akshare as ak
        
        # 判断市场前缀
        if stock_code.startswith(('0', '3')):
            symbol = f'sz{stock_code}'
        elif stock_code.startswith('6'):
            symbol = f'sh{stock_code}'
        else:
            symbol = stock_code
            
        # 获取实时行情
        df = ak.stock_zh_a_spot_em()
        stock_data = df[df['代码'] == stock_code]
        
        if not stock_data.empty:
            return float(stock_data['最新价'].iloc[0])
            
    except Exception as e:
        logger.warning(f"AkShare 接口失败: {e}")
    
    return None

def get_stock_price_sina(stock_code):
    """
    使用新浪财经接口获取股票价格（备用方案）
    """
    try:
        # 构造股票代码
        if stock_code.startswith(('0', '3')):
            symbol = f'sz{stock_code}'
        elif stock_code.startswith('6'):
            symbol = f'sh{stock_code}'
        else:
            symbol = stock_code
            
        url = f'http://hq.sinajs.cn/list={symbol}'
        headers = {
            'Referer': 'http://finance.sina.com.cn',
            'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
        }
        
        response = requests.get(url, headers=headers, timeout=10)
        response.encoding = 'gbk'
        
        if response.status_code == 200:
            data = response.text
            # 解析数据格式: var hq_str_sh601006="大秦铁路, 7.890, 7.920, ...";
            parts = data.split('"')
            if len(parts) > 1:
                price_data = parts[1].split(',')
                if len(price_data) > 3:
                    return float(price_data[3])  # 当前价格
            
    except Exception as e:
        logger.warning(f"新浪接口失败: {e}")
    
    return None

def get_stock_price(stock_code, max_retries=3):
    """
    主获取函数，包含重试机制和多个数据源备选
    """
    data_sources = [
        get_stock_price_tushare,
        get_stock_price_akshare, 
        get_stock_price_sina
    ]
    
    for attempt in range(max_retries):
        for source_func in data_sources:
            try:
                price = source_func(stock_code)
                if price is not None:
                    logger.info(f"成功获取 {stock_code} 价格: {price} (来源: {source_func.__name__})")
                    return price
                    
            except Exception as e:
                logger.warning(f"数据源 {source_func.__name__} 尝试 {attempt+1} 失败: {e}")
                continue
                
        # 所有数据源都失败，等待后重试
        if attempt < max_retries - 1:
            time.sleep(2)  # 等待2秒后重试
    
    logger.error(f"所有数据源均无法获取 {stock_code} 的价格")
    return None

def update_stock_prices(table_name):
    """
    更新股价的主函数
    """
    try:
        rows = base.list_rows(table_name)
        update_count = 0
        error_count = 0
        
        for row in rows:
            stock_code = row.get('股票代码')
            row_id = row.get('_id')
            
            if not stock_code or not isinstance(stock_code, str):
                continue
                
            # 清理股票代码（去除可能的前缀或空格）
            stock_code = stock_code.strip().replace('SH', '').replace('SZ', '')
            
            current_price = get_stock_price(stock_code)
            
            if current_price is not None:
                # 获取昨日收盘价用于计算涨跌幅
                previous_close = row.get('昨日收盘价')
                
                update_data = {
                    '当前股价': round(current_price, 2),
                    '最后更新时间': time.strftime('%Y-%m-%d %H:%M:%S')
                }
                
                # 如果有昨日收盘价，计算涨跌幅
                if previous_close and previous_close > 0:
                    change_percent = (current_price - previous_close) / previous_close * 100
                    update_data['涨跌幅'] = round(change_percent, 2)
                
                base.update_row(table_name, row_id, update_data)
                update_count += 1
                
                # 添加延迟，避免请求过于频繁
                time.sleep(0.5)
                
            else:
                error_count += 1
                logger.error(f"无法获取股票 {stock_code} 的价格")
        
        logger.info(f"更新完成: 成功 {update_count} 条, 失败 {error_count} 条")
        
    except Exception as e:
        logger.error(f"更新股价过程中发生错误: {e}")
        raise

# 配置示例（需要在SeaTable脚本设置中添加环境变量）
"""
在SeaTable脚本设置的"环境变量"中添加：
TUSHARE_TOKEN=你的tushare_pro_token
"""

if __name__ == '__main__':
    try:
        update_stock_prices('股票监控表')  # 替换为你的实际表名
        print("股价更新脚本执行完成")
    except Exception as e:
        print(f"脚本执行失败: {e}")

In [ ]:
def check_price_alert(stock_code, current_price, previous_close, alert_threshold=5):
    """
    检查价格警报
    """
    if previous_close and previous_close > 0:
        change_percent = abs((current_price - previous_close) / previous_close * 100)
        if change_percent >= alert_threshold:
            message = f"🚨 股票 {stock_code} 价格波动 {change_percent:.2f}%！当前价: {current_price}"
            send_alert_notification(message)
            return True
    return False

def send_alert_notification(message):
    """
    发送警报通知（示例：邮件通知）
    """
    # 这里可以集成邮件、微信、钉钉等通知方式
    logger.info(f"警报: {message}")
    # 实际实现可以使用smtplib发邮件，或调用Server酱等微信通知服务